# NLP Assignment 1

### Team Members:
* Muhammad Faizan (1071)
* Muhammad Asim (0840)
* Daud Arif (0833)

---
### Project Overview
The objective of this assignment is to build an end-to-end, reproducible text processing pipeline on a real-world news corpus. Raw text obtained from public news sources is messy and noisy: it contains wire service datelines, social media handles, formatting glitches, non-standard capitalization, and varied linguistic surface forms.

We systematically transition raw news text into an analysis-ready corpus that can directly feed downstream text classification and representation models.


## Task 1: Dataset Setup & Acquisition

### Dataset Selection Rationale
We selected the **McIntire Fake and Real News Corpus** (6,335 articles). 
* **Prediction Target:** Binary classification (`REAL` vs. `FAKE`).
* **Source:** Public research dataset compiling verified news articles from legitimate news wire agencies alongside flagged misinformation, hyper-partisan blog posts, and fake news.
* **Corpus Profile:** Unlike short-text Twitter or headline datasets, this corpus features rich, long-form journalism (averaging 700+ words per article). This depth is ideal for exploring Zipfian distributions, sentence structures, and genuine preprocessing challenges.
* **Licensing & Reproducibility:** Publicly accessible for academic NLP research. We automate downloading via Python so raw data never needs to be manually committed into Git.


In [1]:
# importing all libraries and setting up spacy
import os
import sys
import io
import re
import html
import unicodedata
import zipfile
import urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# nltk stuff for tokenizing and stop words
import nltk
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import spacy

# plot settings so figures dont look ugly
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

print(f"Python Version: {sys.version.split()[0]}")
print(f"Pandas Version: {pd.__version__}")
print(f"NLTK Version: {nltk.__version__}")
print(f"spaCy Version: {spacy.__version__}")


Python Version: 3.10.0
Pandas Version: 2.3.3
NLTK Version: 3.10.4
spaCy Version: 3.8.18


In [2]:
# download helper so we dont have to manually unzip 30mb csv
def acquire_dataset(data_dir="data"):
    # make folders if they dont exist yet
    raw_dir = os.path.join(data_dir, "raw")
    processed_dir = os.path.join(data_dir, "processed")
    os.makedirs(raw_dir, exist_ok=True)
    os.makedirs(processed_dir, exist_ok=True)
    
    csv_target = os.path.join(raw_dir, "fake_or_real_news.csv")
    
    # check if already downloaded so we dont redownload every single time we run kernel
    if not os.path.exists(csv_target):
        print("Fetching corpus archive from public repository...")
        url = "https://raw.githubusercontent.com/joolsa/fake_real_news_dataset/master/fake_or_real_news.csv.zip"
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as response:
            with zipfile.ZipFile(io.BytesIO(response.read())) as z:
                z.extract("fake_or_real_news.csv", raw_dir)
        print(f"Successfully downloaded and extracted to: {csv_target}")
    else:
        print(f"Corpus verified locally at: {csv_target}")
    
    df_raw = pd.read_csv(csv_target)
    # unnamed column is just index from pandas export, rename to article_id
    if "Unnamed: 0" in df_raw.columns:
        df_raw = df_raw.rename(columns={"Unnamed: 0": "article_id"})
    return df_raw

# load the raw data
raw_df = acquire_dataset()
print(f"Total Rows: {len(raw_df):,}")
print(f"Columns: {list(raw_df.columns)}")
display_df = raw_df[["article_id", "title", "label"]].head()
print(display_df.to_string(index=False))


Corpus verified locally at: data\raw\fake_or_real_news.csv
Total Rows: 6,335
Columns: ['article_id', 'title', 'text', 'label']
 article_id                                                         title label
       8476                                  You Can Smell Hillary’s Fear  FAKE
      10294 Watch The Exact Moment Paul Ryan Committed Political Suicide   FAKE
       3608          Kerry to go to Paris in gesture of sympathy          REAL
      10142    Bernie supporters on Twitter erupt in anger against...      FAKE
        875  The Battle of New York: Why This Primary Matters             REAL
